In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
np.random.seed(18)

import tensorflow as tf

from func_file_Model import ResNet_model_paper
from func_file_Data import Normalize_data, Load_sequence

## Load the DAMN model

Tensorflow may print warnings despite operating correctly.

In [ ]:
#Load the DAMN model (ResNet architecture)
model_ResNet = ResNet_model_paper(channels = 64, num_blocks_array = [3,3,3,3], 
                                  kernel_sizes=[5,7,9,11], LeakyReLU_slope=0.1, 
                                  padding="same", interpolation="bilinear", kernel_initializer="he_uniform")

#Compile and load weights
_ = model_ResNet(tf.random.normal([1, 50, 50, 1]))
model_ResNet.load_weights("DAMN_ResNet.keras")

In [ ]:
#Visualize the model summary
model_ResNet.summary()

## Load data to reconstruct

Demonstrating on 10 samples with 128x128 pixels from the tubulin dataset.

In [ ]:
#Load and set data samples as a numpy array of shape [number_of_sample, height, width]
#For a single image, set to [1, height, width]

loaded_data = Load_sequence()   #Load the demo of 10 tubulin frames; replace with targeted data

if len(loaded_data.shape) != 3:
    print("Unexpected data shape.")
print("Data shape:", loaded_data.shape)

By default, the model was trained using 50x50 images normed to a unit sum.<br>
For images of different shapes, a properly adjusted normalization is applied.<br>
Check the Normalize_data function in the func_file_Data.py for details.

In [ ]:
#Normalizing
frames, norms = Normalize_data(loaded_data)

Notes:<br>
-> Tensorflow expects data of shape [number_of_sample, height, width, channels]. Since we use intensity images, channels = 1<br>
-> Tensorflow may print warnings despite working correctly<br>
-> Model initialization may take long the first time<br>

In [ ]:
#Set the evaluation batch_size depending on your available GPU memory
evaluation_batch = 8

#Apply the DAMN model
#Takes approximately 5 seconds on our computational resources + initialization time (up to a minute the first run time)
predicted = np.squeeze(model_ResNet.predict(np.expand_dims(frames, axis=-1), 
                                            batch_size=evaluation_batch, verbose=1))

#Return normalization
reconstructed = predicted * norms[:,None,None]

print("Reconstructed images of shape:", reconstructed.shape)

### Progress with any desired evaluation:

In [ ]:
#Your code goes here